# CompBench Text Prompt Agent Evaluation

Runs the MMEdit agent over selected CompBench tasks using the original text instruction and text-only reference selection. Set `TASKS` in the setup cell to `None` for all tasks or a set such as `{'add', 'replace'}` to filter by `metadata.task`. Annotation boxes, points, and scribbles are not passed to the agent; it selects entities from natural-language descriptions. Set `RESUME_LAST_RUN = True` in the run cell to continue the latest run, skipping completed samples and retrying failed ones; otherwise each execution creates a fresh run. Shared JSONL/Hugging Face image loading, reference preparation, agent execution, and debug capture live in `evaluation/common.py`; task filtering lives in `evaluation/compbench_selection.py`.

Install with `uv sync --extra models --extra benchmark`. This notebook loads `compbench_local_multiobject_mixed.jsonl` from the latest `.out/annotation/compbench/run_*/` by default. Set `MMEDIT_COMPBENCH_ANNOTATION_RUN=run_<timestamp>` to choose a run, or `MMEDIT_COMPBENCH_ANNOTATIONS` to provide an explicit JSONL path. Configure `MMEDIT_API_KEY` or `GOOGLE_API_KEY`. Available tasks are `add`, `remove`, `replace`, `multi_object_add`, and `multi_object_remove`. `START_INDEX` uses original annotation indices and `SAMPLE_LIMIT` applies after filtering. Use `compbench_debug.ipynb` for a representative sample from each task. HF image references use original split row indices.

Results are saved under `.out/evaluation/compbench/text/run_<timestamp>/`.

In [ ]:
import matplotlib.pyplot as plt
from evaluation.notebook import create_run_directory, load_benchmark
from evaluation.common import annotate_sample, load_sample, run_benchmark, select_reference_prompts

BENCHMARK = "compbench"
PROMPT_MODE = "text"
TASKS = None  # Example: {"add", "replace"}; None runs every task.
context = load_benchmark(BENCHMARK, prompt_mode=PROMPT_MODE)
ANNOTATION_FILE = context.annotation_file
OUTPUT_ROOT = context.output_dir
records = context.records
image_loader = context.image_loader
print(f"{len(records)} CompBench annotations loaded from {ANNOTATION_FILE}")
print("Fields:", sorted(records[0]))

In [ ]:
SAMPLE_INDEX = 0
record = records[SAMPLE_INDEX]
sample = load_sample(record, benchmark=BENCHMARK, image_loader=image_loader, prompt_mode=PROMPT_MODE)
if not sample["instruction"]:
    raise ValueError("CompBench record has no instruction.")
print(sample["sample_id"], sample["instruction"])
print("Text references:", [{"token": reference["token"], "refers_to": reference["span"]} for reference in sample["references"]])
select_reference_prompts(sample, prompt_mode=PROMPT_MODE)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(annotate_sample(sample))
axes[0].set_title("Source image (no visual selector in text mode)")
axes[0].axis("off")
if sample["target_image"] is None:
    axes[1].text(0.5, 0.5, "No target image", ha="center", va="center")
else:
    axes[1].imshow(sample["target_image"])
axes[1].set_title("Benchmark target")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from framework.tools import image_editing_tools
from evaluation.notebook import create_agent_model

llm = create_agent_model()
tools = image_editing_tools()
print("CompBench agent ready:", llm.model_id)

In [ ]:
# START_INDEX uses original JSONL indices; SAMPLE_LIMIT applies after task filtering.
from evaluation.compbench_selection import select_compbench_indices
START_INDEX = 0
SAMPLE_LIMIT = None
SAMPLE_INDICES = select_compbench_indices(
    records, tasks=TASKS, start_index=START_INDEX, limit=SAMPLE_LIMIT
)
if not SAMPLE_INDICES:
    raise ValueError("No CompBench annotations match the selected tasks and range.")
print("Selected tasks:", sorted(TASKS) if TASKS else "all")
print(f"Processing {len(SAMPLE_INDICES)} selected annotations.")
from evaluation.notebook import resolve_run_directory
RESUME_LAST_RUN = False  # True continues the latest run and skips completed samples.
RUN_DIR = resolve_run_directory(OUTPUT_ROOT) if RESUME_LAST_RUN else create_run_directory(OUTPUT_ROOT)
benchmark_results = run_benchmark(
    records,
    SAMPLE_INDICES,
    benchmark=BENCHMARK,
    prompt_mode=PROMPT_MODE,
    output_dir=RUN_DIR,
    image_loader=image_loader,
    model=llm,
    tools=tools,
    resume=RESUME_LAST_RUN,
    display_fn=None,
)
print("Run directory:", RUN_DIR)
print(f"Processed {len(benchmark_results)} CompBench samples.")

In [ ]:
completed = [result for result in benchmark_results if result["status"] == "completed"]
failed = [result for result in benchmark_results if result["status"] == "failed"]
print(
    f"CompBench samples: {len(benchmark_results)} | "
    f"completed: {len(completed)} | failed: {len(failed)}"
)
if benchmark_results:
    mean_runtime = sum(result["elapsed_seconds"] for result in benchmark_results) / len(benchmark_results)
    print(f"Mean runtime: {mean_runtime:.2f}s")
print("Compare saved outputs with the benchmark targets; image-quality scoring is separate.")